# Network experiments

Run the setup cell, then choose either experiment. Each experiment writes its results to a separate run directory. [Core workflow](main.ipynb)

In [ ]:
from pathlib import Path
import math

import qoblib_tools as qt
from divi.backends import JobConfig, MaestroSimulator, QoroService
from network_workflow import draw_best_saved, evaluate_layouts, refine_starts, sample_layouts

n_nodes = 11
if not 5 <= n_nodes <= 24:
    raise ValueError("Choose an instance between 5 and 24 routers.")
instance = f"network{n_nodes:02d}"
demand = qt.load_network_demand(n_nodes)
print(instance)
print(demand)
from random import Random

## Search configurations locally

Run the demand cell, the function definitions, then this cell. The first trial uses the starter settings. The remaining trials sample distinct combinations from the lists below. Increase `trial_budget` for a wider search, or set it to `None` to visit the full grid. Set `target_score` to stop as soon as a checked solution reaches that score.

Larger beams and more shots increase sampling work. More starts and swaps spend more time on classical routing. Each CSV row records all settings, the checked score, and elapsed seconds so you can compare quality and cost. Each run saves its winning solution in a fresh directory.

In [40]:
from datetime import datetime
from time import perf_counter
import csv
from sklearn.model_selection import ParameterGrid

# The first value in each list defines the baseline trial.
search_space = {
    "demand_weight": [0.0, 0.25, 0.5, 1.0, 2.0],
    "degree_weight": [2.0, 1.0, 4.0],
    "cluster_size": [8, 6, 10, 12],
    "layers": [1, 2, 3],
    "shots": [2000, 1000, 4000],
    "population_size": [8, 4, 16],
    "iterations": [3, 2, 6],
    "beam_width": [24, 12, 48],
    "partition_candidates": [8, 4, 12],
    "n_starts": [5, 2, 8],
    "swaps_per_start": [200, 50, 400],
}
trial_budget = 32  # Includes the baseline.
search_seed = 7
target_score = None

grid = ParameterGrid(search_space)
trial_count = len(grid) if trial_budget is None else min(trial_budget, len(grid))
if trial_count < 1:
    raise ValueError("Choose a positive trial budget.")
if trial_count == len(grid):
    trial_indices = range(len(grid))
else:
    trial_indices = [0] + Random(search_seed).sample(range(1, len(grid)), trial_count - 1)

reward, penalty, search_arcs = qt.build_network_qubo(demand)
run_dir = Path("starter_results/grid") / instance / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
run_dir.mkdir(parents=True)
print(f"Testing {trial_count:,} of {len(grid):,} configurations. Results: {run_dir}")
winner_score, winner = math.inf, None
with (run_dir / "trials.csv").open("w", newline="") as log:
    writer = csv.DictWriter(log, fieldnames=["trial", *search_space, "seed", "checked_score", "seconds"])
    writer.writeheader()
    for trial, index in enumerate(trial_indices, start=1):
        config = grid[index]
        start = perf_counter()
        print(f"Trial {trial}/{trial_count}: {config}", flush=True)
        model = config["demand_weight"] * reward + config["degree_weight"] * penalty
        samples = sample_layouts(
            model, MaestroSimulator(shots=config["shots"]), config["cluster_size"], config["layers"],
            search_seed, population_size=config["population_size"], iterations=config["iterations"],
            beam_width=config["beam_width"], partition_candidates=config["partition_candidates"],
        )
        scores, *_ = evaluate_layouts(samples, model, search_arcs, demand)
        score, result = refine_starts(scores, demand, config["n_starts"], config["swaps_per_start"], search_seed)
        if result is not None and score < winner_score:
            winner_score, winner = score, config
            qt.write_network_solution(run_dir / f"{instance}.sol", len(demand), result)
        writer.writerow(dict(config, trial=trial, seed=search_seed, checked_score=score,
                             seconds=round(perf_counter() - start, 2)))
        log.flush()
        if target_score is not None and winner_score <= target_score:
            break

if winner is None:
    print("No routable candidate was found.")
else:
    print(f"Best checked score: {winner_score:,} with {winner}")
print("Results directory:", run_dir)

Testing 32 of 393,660 configurations. Results: starter_results/grid/network11/20261121-201118-017152
Trial 1/32: {'swaps_per_start': 200, 'shots': 2000, 'population_size': 8, 'partition_candidates': 8, 'n_starts': 5, 'layers': 1, 'iterations': 3, 'demand_weight': 0.0, 'degree_weight': 2.0, 'cluster_size': 8, 'beam_width': 24}
Start   Initial LP   Accepted   Checked score
    1      306,750         12         259,118
    2      316,500          9         264,700
    3      345,000         14         257,539
    4      352,667          9         260,800
    5      352,667          7         258,750
Trial 2/32: {'swaps_per_start': 200, 'shots': 4000, 'population_size': 8, 'partition_candidates': 8, 'n_starts': 8, 'layers': 3, 'iterations': 2, 'demand_weight': 0.5, 'degree_weight': 2.0, 'cluster_size': 6, 'beam_width': 12}
Start   Initial LP   Accepted   Checked score
    1      316,167         13         262,200
    2      321,750         11         262,700
    3      328,800          6  

## Test routing feedback and alternative swaps

Run setup above, then this cell. Compare the baseline, each change by itself, and all five together. Each seed starts with a shared direct-demand QUBO round. In round two the path variants change the QUBO. The LP feedback variant uses shadow prices from the best first-round layout to reward direct shortcuts around congested two-link routes.

The two-hop term rewards a pair of installed links in proportion to the traffic between its endpoints. The concentration term estimates how much that traffic would increase the squared load on both legs if it joined their direct traffic. Both are proxies. The routing LP determines the actual loads.

Refine up to three starts with 200 legal proposals per start. Annealing accepts occasional uphill moves while retaining the best visited layout. The three-edge option mixes two-link swaps with cyclic destination swaps among three distinct sources and destinations. Each finalist receives an integer routing and a checker score. The CSV records scores before and after swaps; compare medians across seeds.

In [42]:
from datetime import datetime
from statistics import median
import csv
import math
from tempfile import TemporaryDirectory

from topology_experiments import improve_layout, topology_model

experiment_dir = Path("starter_results/topology_experiments") / instance / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
experiment_dir.mkdir(parents=True)
seeds = [7, 11, 23]
variants = ["baseline", "two-hop", "concentration", "LP feedback", "annealing", "three-edge", "combined"]
experiment_backend = MaestroSimulator(shots=2000)

def propose(demand, seed, **model_options):
    model, arcs = topology_model(demand, **model_options)
    samples = sample_layouts(model, experiment_backend, 8, 1, seed, beam_width=24)
    scores, *_ = evaluate_layouts(samples, model, arcs, demand)
    return {layout: score for layout, score in scores.items() if math.isfinite(score)}

results = {variant: [] for variant in variants}
lowest = math.inf
with (experiment_dir / "trials.csv").open("w", newline="") as log, TemporaryDirectory() as scratch:
    writer = csv.writer(log)
    writer.writerow(["variant", "seed", "starts", "pre_swap_lp", "checked_score"])
    for seed in seeds:
        first = propose(demand, seed)
        if not first:
            raise RuntimeError(f"No routable first-round candidate for seed {seed}")
        incumbent = min(first, key=first.get)
        # All variants share round one. Swap-only variants also share round two.
        plain = propose(demand, seed + 100)
        for variant in variants:
            if variant in ("baseline", "annealing", "three-edge"):
                second = plain
            else:
                second = propose(demand, seed + 100,
                                 two_hop=variant in ("two-hop", "combined"),
                                 concentration=variant in ("concentration", "combined"),
                                 feedback_layout=incumbent if variant in ("LP feedback", "combined") else None)
            pool = first | second
            starts = sorted(pool, key=lambda layout: (pool[layout], layout))[:3]
            final_score = math.inf
            for rank, layout in enumerate(starts):
                improved, _ = improve_layout(demand, layout, pool[layout], seed + rank,
                                             trials=200, anneal=variant in ("annealing", "combined"),
                                             three_swap=variant in ("three-edge", "combined"))
                routing = qt.route_integral(demand, improved)
                tmp = Path(scratch) / f"{instance}.sol"
                qt.write_network_solution(tmp, n_nodes, routing)
                valid, score, reason = qt.check_network_solution(tmp, n_nodes, demand)
                if not valid:
                    raise ValueError(reason)
                final_score = min(score, final_score)
                if score < lowest:
                    lowest = score
                    qt.write_network_solution(experiment_dir / f"{instance}.sol", n_nodes, routing)
            results[variant].append(final_score)
            writer.writerow([variant, seed, len(starts), min(pool.values()), final_score])
            log.flush()
            print(variant, seed, final_score, flush=True)
for variant, values in results.items():
    print(f"{variant:15s} median {median(values):,.0f}   best {min(values):,}")
print("Saved in", experiment_dir)

baseline 7 257364
two-hop 7 253834
concentration 7 255200
LP feedback 7 246875
annealing 7 257500
three-edge 7 254100
combined 7 259250
baseline 11 252858
two-hop 11 256667
concentration 11 265286
LP feedback 11 260286
annealing 11 257875
three-edge 11 251000
combined 11 260834
baseline 23 249400
two-hop 23 250375
concentration 23 250375
LP feedback 23 258200
annealing 23 248910
three-edge 23 246429
combined 23 254715
baseline        median 252,858   best 249,400
two-hop         median 253,834   best 250,375
concentration   median 255,200   best 250,375
LP feedback     median 258,200   best 246,875
annealing       median 257,500   best 248,910
three-edge      median 251,000   best 246,429
combined        median 259,250   best 254,715
Saved in starter_results/topology_experiments/network11/20261121-201242-905009
